In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_underlaw_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #      "UnderLawId string,"
        #      "SourceSystem string,"
        #      "PortfolioKey string,"
        #      "PortfolioName string,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId string,"
        #      "UnderLawKey string,"
        #      "CDI string,"
        #      "TaxRegistrationNumber string,"
        #      "ApplicationNumber string,"
        #      "ApplicationStatusCode string,"
        #      "ApplicationStatus string,"
        #      "ApplicationStatusDate timestamp,"
        #      "TypeOfCourt string,"
        #      "CourtDate date,"
        #      "RecordDate date,"
        #      "SubmissionDate date,"
        #      "InitiationDate date,"
        #      "CourtOutcomeCode string,"
        #      "CourtOutcome string,"
        #      "CompletionReasonCode int,"
        #      "CompletionReason string,"
        #      "LawSystem string,"
        #      "PlatformName string,"
        #      "InLawFlag int,"
        #      "LawStatusCode string,"
        #      "LawStatus string,"
        #      "CalcLawStatusCode string,"
        #      "CalcLawStatus string,"
        #      "UnderLawType string,"
        #      "UnderLawTypeCode string,"
        #      "TemporaryCourtDecisionDate timestamp,"
        #      "DeductionDate timestamp,"
        #      "CountyCourtCode string,"
        #      "CountyCourt string,"
        #      "InterbankStatus3869Code string,"
        #      "InterbankStatus3869 string,"
        #      "SubmissionDateGroup string,"
        #      "ResubmissionStatusFlag string,"
        #      "ResubmissionStatusCode string,"
        #      "ResubmissionStage string,"
        #      "AppealCourtDate timestamp,"
        #      "CepalEligibilityDate timestamp,"
        #      "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.UnderLaw"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================ InLawFlagN3869 ===============

        inlawflagn3869_extra_filters = None

        if inlawflagn3869_extra_filters is not None:
            inlawflagn3869_extra_filters.append(date_filter)
            inlawflagn3869_final_filters = " AND ".join(inlawflagn3869_extra_filters)
        else:
            inlawflagn3869_final_filters = date_filter

        inlawflagn3869_selected_fields = [
            "Description",
            "InLawFlagId",
            "InLawFlag",
            "ApplicationStatus",
            "CalcLawStatus",
        ]

        inlawflagn3869 = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "params_inlawflagn3869",
            inlawflagn3869_final_filters,
            inlawflagn3869_selected_fields,
        )

        # ============ fs_customers_snap =============

        fs_customers_snap_extra_filters = None

        if fs_customers_snap_extra_filters is not None:
            fs_customers_snap_extra_filters.append(date_filter)
            fs_customers_snap_final_filters = " AND ".join(
                fs_customers_snap_extra_filters
            )
        else:
            fs_customers_snap_final_filters = date_filter

        fs_customers_snap_selected_fields = [
            "CUST_CODE",
            "CUCO_DEDICATED_PROPERTY_08_INTCODE",
            "CUCO_DEDICATED_PROPERTY_08_INTCODE_DATE",
            "CUCO_DEDICATED_PROPERTY_03_INTCODE",
            "CUST_SLC_ID",
            "BusinessDate",
        ]

        fs_customers_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_customers_snap",
            fs_customers_snap_final_filters,
            fs_customers_snap_selected_fields,
        )

        # ============ fs_customers_snap_extra =============

        fs_customers_snap_extra_extra_filters = None

        if fs_customers_snap_extra_extra_filters is not None:
            fs_customers_snap_extra_extra_filters.append(date_filter)
            fs_customers_snap_extra_final_filters = " AND ".join(
                fs_customers_snap_extra_extra_filters
            )
        else:
            fs_customers_snap_extra_final_filters = date_filter

        fs_customers_snap_extra_selected_fields = [
            "CUST_CODE",
            "CUSX_GENERIC_DATETIME_05",
            "CUSX_GENERIC_DATETIME_06",
        ]

        fs_customers_snap_extra = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_customers_snap_extra",
            fs_customers_snap_extra_final_filters,
            fs_customers_snap_extra_selected_fields,
        )

        # ============== FSAD_CUSTOMERS_GENERIC_OPEN ==========

        fsad_customers_generic_open_extra_filters = None

        if fsad_customers_generic_open_extra_filters is not None:
            fsad_customers_generic_open_extra_filters.append(date_filter)
            fsad_customers_generic_open_final_filters = " AND ".join(
                fsad_customers_generic_open_extra_filters
            )
        else:
            fsad_customers_generic_open_final_filters = date_filter

        fsad_customers_generic_open_selected_fields = [
            "CUST_CODE",
            "CUSG_GENERIC_DATETIME_05",
            "CUSG_GENERIC_DATETIME_06",
            "CUSG_GENERIC_DATETIME_07",
            "CUSG_GENERIC_NVARCHAR_16",
            "CUSG_GENERIC_NVARCHAR_19",
            "CUSG_GENERIC_NVARCHAR_20",
            "CUSG_GENERIC_NVARCHAR_24",
        ]

        fsad_customers_generic_open = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_customers_generic_open",
            fsad_customers_generic_open_final_filters,
            fsad_customers_generic_open_selected_fields,
        )

        # =============== dim_customer_static =============

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = ["cust_id", "cust_cif"]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ======================== fsad_list_of_values ===================
        fsad_list_of_values_extra_filters = None

        if fsad_list_of_values_extra_filters is not None:
            fsad_list_of_values_extra_filters.append(date_filter)
            fsad_list_of_values_final_filters = " AND ".join(
                fsad_list_of_values_extra_filters
            )
        else:
            fsad_list_of_values_final_filters = date_filter

        fsad_list_of_values_selected_fields = [
            "LOV_DESC",
            "LOV_CODE",
            "LOVT_CODE",
            "LOVT_DESC",
            "LOV_COMMENTS",
        ]

        fsad_list_of_values = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_list_of_values",
            fsad_list_of_values_final_filters,
            fsad_list_of_values_selected_fields,
        )

        fsad_list_of_values_1 = fsad_list_of_values
        fsad_list_of_values_2 = (
            fsad_list_of_values_1.withColumnRenamed("LOVT_CODE", "LOVT_CODE_2")
            .withColumnRenamed("LOV_DESC", "LOV_DESC_2")
            .withColumnRenamed("LOV_COMMENTS", "LOV_COMMENTS_2")
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ==================== joins ====================

        underlaw_df = (
            fs_customers_snap.join(
                fsad_customers_generic_open,
                (
                    fs_customers_snap["CUST_CODE"]
                    == fsad_customers_generic_open["CUST_CODE"]
                ),
                "inner",
            )
            .join(
                fs_customers_snap_extra,
                (
                    fs_customers_snap["CUST_CODE"]
                    == fs_customers_snap_extra["CUST_CODE"]
                ),
                "inner",
            )
            .join(
                inlawflagn3869,
                (
                    inlawflagn3869["InLawFlagId"]
                    == fs_customers_snap["CUCO_DEDICATED_PROPERTY_08_INTCODE"]
                ),
                "inner",
            )
            .join(
                dim_customer_static,
                (
                    (dim_customer_static["cust_id"] == fs_customers_snap["CUST_CODE"])
                    & (fs_customers_snap["CUST_SLC_ID"] == "ALL")
                ),
                "inner",
            )
            .join(
                portfolio,
                (
                    substring(dim_customer_static["cust_cif"], 1, 3)
                    == portfolio["prefix"]
                ),
                "left",
            )
            .join(
                fsad_list_of_values,
                (
                    (
                        fsad_list_of_values["LOV_CODE"]
                        == fs_customers_snap["CUCO_DEDICATED_PROPERTY_08_INTCODE"]
                    )
                    & (fsad_list_of_values["LOVT_CODE"] == 839)
                ),
                "left",
            )
            .join(
                fsad_list_of_values_2,
                (
                    (
                        fsad_list_of_values_2["LOV_CODE"]
                        == fs_customers_snap["CUCO_DEDICATED_PROPERTY_03_INTCODE"]
                    )
                    & (fsad_list_of_values_2["LOVT_CODE_2"] == 834)
                ),
                "left",
            )
        )

        # =========== underlaw_df =================

        UnderLawId_primary_key_fields = [
            dim_customer_static.cust_cif,
            lit("3869"),
            lit("qca"),
        ]

        UnderLawKey_primary_key_fields = [dim_customer_static.cust_cif, lit("3869")]

        underlaw_df = (
            underlaw_df.withColumn(
                "UnderLawId",
                sha2(
                    concat_ws(";", *UnderLawId_primary_key_fields),
                    256,
                ),
            )
            .withColumn("UnderLawKey", concat_ws(";", *UnderLawKey_primary_key_fields))
            .withColumn(
                "InLawFlag",
                when(inlawflagn3869.InLawFlag == "Εντός", 1)
                .when(inlawflagn3869.InLawFlag == "Εκτός", 0)
                .otherwise(None),
            )
        )

        # ================== Selection of Final Columns ==================

        underlaw_qca = underlaw_df.select(
            underlaw_df.UnderLawId.alias("UnderLawId"),
            lit("qca").alias("SourceSystem"),
            coalesce(portfolio.portfolio_id_qca, lit(2))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Retail"))
            .cast("string")
            .alias("PortfolioName"),            
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            underlaw_df.UnderLawKey.alias("UnderLawKey"),
            dim_customer_static.cust_cif.cast("string").alias("CDI"),
            lit(None).alias("TaxRegistrationNumber"),
            lit(None).alias("ApplicationNumber"),
            inlawflagn3869.ApplicationStatus.alias("ApplicationStatusCode"),
            inlawflagn3869.ApplicationStatus.alias("ApplicationStatus"),
            fs_customers_snap.CUCO_DEDICATED_PROPERTY_08_INTCODE_DATE.cast("timestamp").alias("ApplicationStatusDate"),
            lit(None).alias("TypeOfCourt"),
            fsad_customers_generic_open.CUSG_GENERIC_DATETIME_07.cast("date").alias(
                "CourtDate"
            ),
            lit(None).alias("RecordDate"),
            lit(None).cast("date").alias("InitiationDate"),
            fs_customers_snap.CUCO_DEDICATED_PROPERTY_08_INTCODE.cast("string").alias(
                "CourtOutcomeCode"
            ),
            inlawflagn3869.Description.cast("string").alias("CourtOutcome"),
            lit(None).alias("CompletionReasonCode"),
            lit(None).alias("CompletionReason"),
            lit(None).alias("LawSystem"),
            lit(None).alias("PlatformName"),
            underlaw_df.InLawFlag.cast("int").alias("InLawFlag"),
            fs_customers_snap.CUCO_DEDICATED_PROPERTY_08_INTCODE.cast("string").alias(
                "LawStatusCode"
            ),
            fsad_list_of_values.LOV_DESC.cast("string").alias("LawStatus"),
            inlawflagn3869.CalcLawStatus.alias("CalcLawStatusCode"),
            inlawflagn3869.CalcLawStatus.alias("CalcLawStatus"),
            fsad_customers_generic_open.CUSG_GENERIC_DATETIME_05.cast("date").alias(
                "SubmissionDate"
            ),
            lit("N3869").alias("UnderLawType"),
            lit("N3869").alias("UnderLawTypeCode"),
            fsad_customers_generic_open.CUSG_GENERIC_DATETIME_06.alias(
                "TemporaryCourtDecisionDate"
            ),
            fs_customers_snap_extra.CUSX_GENERIC_DATETIME_05.alias("DeductionDate"),
            fsad_customers_generic_open.CUSG_GENERIC_NVARCHAR_16.alias(
                "CountyCourtCode"
            ),
            fsad_customers_generic_open.CUSG_GENERIC_NVARCHAR_16.alias("CountyCourt"),
            fs_customers_snap.CUCO_DEDICATED_PROPERTY_03_INTCODE.cast("string").alias(
                "InterbankStatus3869Code"
            ),
            fsad_list_of_values_2.LOV_COMMENTS_2.cast("string").alias(
                "InterbankStatus3869"
            ),
            fsad_customers_generic_open.CUSG_GENERIC_NVARCHAR_19.alias(
                "SubmissionDateGroup"
            ),
            fsad_customers_generic_open.CUSG_GENERIC_NVARCHAR_20.alias(
                "ResubmissionStatusFlag"
            ),
            fsad_customers_generic_open.CUSG_GENERIC_NVARCHAR_24.alias(
                "ResubmissionStatusCode"
            ),
            fsad_customers_generic_open.CUSG_GENERIC_NVARCHAR_24.alias(
                "ResubmissionStage"
            ),
            fs_customers_snap_extra.CUSX_GENERIC_DATETIME_06.alias("AppealCourtDate"),
            lit(None).alias("CepalEligibilityDate"),
            fs_customers_snap.BusinessDate.alias("BusinessDate"),
        )

        # ========================== Append ==========================
        underlaw_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_underlaw_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)
